# Tools from an MCP Server

**What you will build:** an agent whose tools come from an **MCP server** instead of your own Python functions. MCP (the Model Context Protocol) is a standard way to expose tools, so a server someone else built plugs straight into your agent, no glue code required.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/18_mcp_tools.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# MCP support needs the extra. Run once.
!pip install -q "pydantic-ai-slim[openai,mcp]>=2.0,<3.0"

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(MODEL_NAME, provider=OpenAIProvider(
    base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"]))
print("Ready:", MODEL_NAME)

## Why MCP

In 1.2 you wrote tools as Python functions. That's perfect for *your* logic — but for common capabilities (search a wiki, query a database, hit GitHub) you'd rather not re-implement and maintain each one. **MCP** standardizes this: a *server* exposes tools over a protocol, and any MCP-aware agent can use them. Write once, use from any agent; or consume servers others maintain.

```{note}
MCP is why it's the loudest "is this course current?" signal in 2026 — both PydanticAI and LangChain ship first-class MCP support. The mechanics are exactly the tool calling you already know (0.0); MCP just changes *where the tools come from*.
```

## Connect to a public MCP server

We'll use **[DeepWiki](https://mcp.deepwiki.com/mcp)** — a free, keyless, hosted MCP server that answers questions about public GitHub repositories. No signup, no Node.js, just a URL. In PydanticAI you wrap it in an `MCPToolset` and hand it to the agent via `toolsets=`; the connection is opened by `async with agent:`.

In [ ]:
from pydantic_ai.mcp import MCPToolset

deepwiki = MCPToolset("https://mcp.deepwiki.com/mcp")   # keyless public MCP server

agent = Agent(
    model,
    toolsets=[deepwiki],
    system_prompt="You answer questions about GitHub repositories using the DeepWiki tools.",
)

# `async with agent:` opens the MCP connection for the duration of the block.
async with agent:
    result = await agent.run("What is the pydantic/pydantic-ai repository for, in two sentences?")
print(result.output)

The agent just used tools it never defined — they live on the DeepWiki server. From the agent's side, an MCP tool is indistinguishable from a `@agent.tool_plain` function: same tool-calling loop, same everything. You gained a capability without writing or maintaining it.

::::{dropdown} 🔍 Other transports, and the LangChain equivalent
:color: info

MCP servers come in two flavours: **HTTP** (like DeepWiki above — just a URL) and **stdio** (a local command, e.g. `npx -y @modelcontextprotocol/server-everything`, wrapped in a `StdioTransport`). Both attach the same way, via `toolsets=[...]`.

**LangChain/LangGraph** consume MCP through `langchain-mcp-adapters`:

```python
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
client = MultiServerMCPClient({"deepwiki": {"transport": "streamable_http", "url": "https://mcp.deepwiki.com/mcp"}})
tools = await client.get_tools()
agent = create_agent(llm, tools)
```
::::

```{warning}
MCP is young and moving fast. This notebook is written for **pydantic-ai 2.2** (`MCPToolset` + `toolsets=` + `async with agent:`). Earlier versions used `MCPServerStdio`/`MCPServerStreamableHTTP` classes that were **removed** in v2.0 — if you find those in an old tutorial, it's outdated. Pin your versions (see the README) and check the [MCP client docs](https://pydantic.dev/docs/ai/mcp/client/) when in doubt.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Ask about another repo.** Point the question at `langchain-ai/langgraph` or a repo of your own and compare answers.
2. **Mix tool sources.** Add a normal `@agent.tool_plain` of your own alongside the MCP toolset — the agent can't tell them apart.
3. **Read the tools.** Inside the `async with`, the agent exposes the MCP tool names; print the run's tool-calls (as in 1.2 / 19) to see which DeepWiki tools it chose.
::::

**What's next:** you just *used* an MCP server; in **18b** you'll *build* one with FastMCP. Then **19** builds the habit of debugging agents, which closes Block 1.